# Deep Neural Network Models for Advanced NLP Problems

# Question Answering System
Albane COIFFE  
Louise LAVERGNE  
Maelwenn LABIDURIE  

In [ ]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

/Users/albanecoiffe/Downloads/NLP/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#data_path = "/content/drive/MyDrive/data_nlp"
data_path = "data/"

## Pre processing

In [3]:
import json
import re
from collections import Counter

In [4]:
def tokenize(text):
    text = text.lower()
    tokens = text.split()
    return tokens

**Dataset JSON Structure**
```
{"version": "v2.0",
 "data": [
	{"title": "TITLE_1",
	"paragraphs": [
		{"qas": [
			{"question": "QUESTION_1",
			"id": ID_1,
			"answers": [
				{"text": "ANSWER_1_1",
				"answer_start": INDEX_1_1},
				{"text": "ANSWER_1_2",
				"answer_start": INDEX_1_2}
			],
			"is_impossible": (false/true)},
			{"plausible_answers": [
				{"text": "P_ANSWER_1_1", "answer_start": P_INDEX_1_1},
				{"text": "P_ANSWER_1_2", "answer_start": P_INDEX_1_2}
			]},
			{"question": “QUESTION_2”,
			"id": ID_2,
			"answers": [
				...
			],
			"is_impossible": (false/true) },
			{"plausible_answers": [
				...
			]}
		],
		"context": "CONTEXT_1"
		},
		{"qas": [
			…
		],
		"context": "CONTEXT_2"
		},
		…
	]}
]}
```



In [5]:
def load_squad(path):
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    examples = []
    for article in squad["data"]: # get data
        for paragraph in article["paragraphs"]: # iterate on paragraphs
            context = paragraph["context"] # get the context
            context_tokens = tokenize(context)
            for qa in paragraph["qas"]: # iterate on the questions
                answer_obj = None
                # get answers if their is at least 1
                if len(qa.get("answers", [])) > 0:
                    answer_obj = qa["answers"][0]
                # if not answers get plausible_answers
                elif len(qa.get("plausible_answers", [])) > 0:
                    answer_obj = qa["plausible_answers"][0]

                # if so answer skip question
                if answer_obj is None: continue

                q_tokens = tokenize(qa["question"])

                char_count = 0
                start_token = end_token = None
                for i, token in enumerate(context_tokens): # iterate on token (word) context
                    if char_count <= answer_obj["answer_start"] < char_count + len(token) + 1:
                        # if index of start answer between current char_count and start of next word
                        start_token = i # start token = index of current token
                    if char_count <= answer_obj["answer_start"] + len(answer_obj["text"]) <= char_count + len(token) + 1:
                      # if end of answer between current char_count and start of next word
                        end_token = i # end token = index of current token
                        break # stop loop
                    char_count += len(token) + 1 # chart_count = current char_count + lenght next word + space

                if start_token is not None and end_token is not None:
                  # if their is a start and end token, add to examples
                    examples.append({
                        "context_tokens": context_tokens,
                        "question_tokens": q_tokens,
                        "start": start_token,
                        "end": end_token,
                        "raw_answer": answer_obj["text"]
                    })
    return examples

In [6]:
data = data_path+"train-v2.0.json"
ex = load_squad(data)

In [7]:
ex[0]

{'context_tokens': ['beyoncé',
  'giselle',
  'knowles-carter',
  '(/biːˈjɒnseɪ/',
  'bee-yon-say)',
  '(born',
  'september',
  '4,',
  '1981)',
  'is',
  'an',
  'american',
  'singer,',
  'songwriter,',
  'record',
  'producer',
  'and',
  'actress.',
  'born',
  'and',
  'raised',
  'in',
  'houston,',
  'texas,',
  'she',
  'performed',
  'in',
  'various',
  'singing',
  'and',
  'dancing',
  'competitions',
  'as',
  'a',
  'child,',
  'and',
  'rose',
  'to',
  'fame',
  'in',
  'the',
  'late',
  '1990s',
  'as',
  'lead',
  'singer',
  'of',
  'r&b',
  'girl-group',
  "destiny's",
  'child.',
  'managed',
  'by',
  'her',
  'father,',
  'mathew',
  'knowles,',
  'the',
  'group',
  'became',
  'one',
  'of',
  'the',
  "world's",
  'best-selling',
  'girl',
  'groups',
  'of',
  'all',
  'time.',
  'their',
  'hiatus',
  'saw',
  'the',
  'release',
  'of',
  "beyoncé's",
  'debut',
  'album,',
  'dangerously',
  'in',
  'love',
  '(2003),',
  'which',
  'established',
  'her

In [8]:
ex[0]['context_tokens'][39]

'in'

In [9]:
ex[0]['context_tokens'][42]

'1990s'

We have the texts tokenize, and the answer format to get the start and end token of each questions.

Now we have to transform the token (word) into int so the model can learn them.

In [10]:
from collections import Counter
# Counter is a dict class where the keys are the word and the values are their count

def build_vocab(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex["context_tokens"])
        counter.update(ex["question_tokens"])
        # update the counter to add 1 to the count

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)
            # add token if at least 2 (default value) count in the text, value equal lenght of vocab (so last added work + 1)

    return vocab


In [11]:
vocab = build_vocab(ex)

Now lets do the final task before the training

In [13]:
import numpy as np

MAX_CONTEXT_LEN = 300
MAX_QUESTION_LEN = 40

In [24]:
def encode(tokens, vocab, max_len):
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    ids = ids[:max_len]
    return ids + [vocab["<PAD>"]] * (max_len - len(ids))

In [25]:
def prepare_data(examples, vocab):
    X_context = []
    X_question = []
    y_start = []
    y_end = []

    for ex in examples:
        if ex["start"] >= MAX_CONTEXT_LEN:
            continue

        X_context.append(encode(ex["context_tokens"], vocab, MAX_CONTEXT_LEN))
        X_question.append(encode(ex["question_tokens"], vocab, MAX_QUESTION_LEN))
        y_start.append(ex["start"])
        y_end.append(ex["end"])

    return (
        np.array(X_context),
        np.array(X_question),
        np.array(y_start),
        np.array(y_end),
    )


In [26]:
X_context, X_question, y_start, y_end = prepare_data(ex, vocab)

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader



In [28]:
class QAModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_units, max_context_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.lstm_units = lstm_units
        self.max_context_len = max_context_len
        
        # Shared embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Bidirectional LSTMs for context and question
        self.lstm_context = nn.LSTM(
            embed_dim, lstm_units, bidirectional=True, batch_first=True
        )
        self.lstm_question = nn.LSTM(
            embed_dim, lstm_units, bidirectional=True, batch_first=True
        )
        
        # Dense layers for start and end predictions
        self.dense_start = nn.Linear(lstm_units * 4, 1)
        self.dense_end = nn.Linear(lstm_units * 4, 1)
    
    def forward(self, context, question):
        # Embed inputs
        context_emb = self.embedding(context)
        question_emb = self.embedding(question)
        
        # Encode with LSTMs
        context_enc, _ = self.lstm_context(context_emb)
        question_enc, _ = self.lstm_question(question_emb)
        
        # Get last output from question encoding
        question_rep = question_enc[:, -1, :].unsqueeze(1)
        
        # Repeat for each context token
        question_rep = question_rep.expand(-1, self.max_context_len, -1)
        
        # Concatenate context and question encodings
        merged = torch.cat([context_enc, question_rep], dim=-1)
        
        # Predict start and end positions
        start_logits = self.dense_start(merged).squeeze(-1)
        end_logits = self.dense_end(merged).squeeze(-1)
        
        return start_logits, end_logits


VOCAB_SIZE = len(vocab)
EMBED_DIM = 100
LSTM_UNITS = 128

model = QAModel(VOCAB_SIZE, EMBED_DIM, LSTM_UNITS, MAX_CONTEXT_LEN)
model.to(device)

print(model)


QAModel(
  (embedding): Embedding(181802, 100, padding_idx=0)
  (lstm_context): LSTM(100, 128, batch_first=True, bidirectional=True)
  (lstm_question): LSTM(100, 128, batch_first=True, bidirectional=True)
  (dense_start): Linear(in_features=512, out_features=1, bias=True)
  (dense_end): Linear(in_features=512, out_features=1, bias=True)
)


In [29]:
BATCH_SIZE = 32
NUM_EPOCHS = 10

# Préparer les données
X_context_tensor = torch.LongTensor(X_context)
X_question_tensor = torch.LongTensor(X_question)
y_start_tensor = torch.LongTensor(y_start)
y_end_tensor = torch.LongTensor(y_end)

# Créer DataLoader
dataset = TensorDataset(X_context_tensor, X_question_tensor, y_start_tensor, y_end_tensor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Définir optimizer et loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Boucle d'entraînement
model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for context, question, start_pos, end_pos in train_loader:
        context = context.to(device)
        question = question.to(device)
        start_pos = start_pos.to(device)
        end_pos = end_pos.to(device)
        
        # Forward pass
        start_logits, end_logits = model(context, question)
        
        # Calculer la loss
        loss_start = criterion(start_logits, start_pos)
        loss_end = criterion(end_logits, end_pos)
        loss = loss_start + loss_end
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/10, Loss: 7.9407
Epoch 2/10, Loss: 7.0633
Epoch 3/10, Loss: 6.5444
Epoch 4/10, Loss: 6.1566
Epoch 5/10, Loss: 5.8549
Epoch 6/10, Loss: 5.6170
Epoch 7/10, Loss: 5.4329
Epoch 8/10, Loss: 5.2842
Epoch 9/10, Loss: 5.1671
Epoch 10/10, Loss: 5.0709


In [33]:
torch.save(model.state_dict(), "bidaf_model.pt")
print("Modèle sauvegardé !")


Modèle sauvegardé !


In [34]:
def predict_answer(model, context_tokens, question_tokens, vocab):
    model.eval()
    with torch.no_grad():
        ctx = torch.LongTensor([encode(context_tokens, vocab, MAX_CONTEXT_LEN)]).to(device)
        qst = torch.LongTensor([encode(question_tokens, vocab, MAX_QUESTION_LEN)]).to(device)
        
        start_logits, end_logits = model(ctx, qst)
        
        start = torch.argmax(start_logits[0]).item()
        end = torch.argmax(end_logits[0]).item()
        
        if end < start:
            end = start
    
    return " ".join(context_tokens[start:end+1])


In [35]:
example = ex[0]
print(predict_answer(model, example["context_tokens"], example["question_tokens"], vocab))


dangerously in love (2003),


## Évaluation sur plusieurs exemples

In [36]:
# Faire des prédictions sur 10 exemples et afficher les résultats
num_examples = min(10, len(ex))
correct = 0

print(f"{'='*100}")
print(f"ÉVALUATION SUR {num_examples} EXEMPLES")
print(f"{'='*100}\n")

for idx in range(num_examples):
    example = ex[idx]
    question_text = " ".join(example["question_tokens"])
    context_text = " ".join(example["context_tokens"])
    true_answer = example["raw_answer"]
    
    # Prédiction du modèle
    predicted_answer = predict_answer(model, example["context_tokens"], example["question_tokens"], vocab)
    
    # Vérification si la prédiction est correcte
    is_correct = predicted_answer.lower() == true_answer.lower()
    correct += is_correct
    status = "✓ CORRECT" if is_correct else "✗ INCORRECT"
    
    print(f"Exemple {idx + 1}:")
    print(f"Question: {question_text}")
    print(f"Contexte: {context_text[:100]}..." if len(context_text) > 100 else f"Contexte: {context_text}")
    print(f"Réponse correcte: {true_answer}")
    print(f"Prédiction: {predicted_answer}")
    print(f"Résultat: {status}")
    print(f"{'-'*100}\n")

print(f"{'='*100}")
print(f"SCORE: {correct}/{num_examples} ({100*correct/num_examples:.1f}%)")
print(f"{'='*100}")

ÉVALUATION SUR 10 EXEMPLES

Exemple 1:
Question: when did beyonce start becoming popular?
Contexte: beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ bee-yon-say) (born september 4, 1981) is an american si...
Réponse correcte: in the late 1990s
Prédiction: dangerously in love (2003),
Résultat: ✗ INCORRECT
----------------------------------------------------------------------------------------------------

Exemple 2:
Question: what areas did beyonce compete in when she was growing up?
Contexte: beyoncé giselle knowles-carter (/biːˈjɒnseɪ/ bee-yon-say) (born september 4, 1981) is an american si...
Réponse correcte: singing and dancing
Prédiction: late 1990s as lead singer of r&b girl-group destiny's child. managed by her father, mathew knowles, the group became one of the world's best-selling girl groups of all time. their hiatus saw the release of beyoncé's debut album, dangerously in love (2003),
Résultat: ✗ INCORRECT
-------------------------------------------------------------------------

## Analyse détaillée des erreurs

In [37]:
def compute_exact_match(prediction, true_answer):
    """Calcule si la prédiction correspond exactement à la réponse"""
    return prediction.lower().strip() == true_answer.lower().strip()

def compute_f1_tokens(prediction, true_answer):
    """Calcule le F1 score basé sur l'intersection de tokens"""
    pred_tokens = set(prediction.lower().split())
    true_tokens = set(true_answer.lower().split())
    
    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    
    common = pred_tokens.intersection(true_tokens)
    if len(common) == 0:
        return 0.0
    
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(true_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Analyser en détail 10 exemples
num_examples = min(10, len(ex))
exact_matches = 0
f1_scores = []

print(f"\n{'='*120}")
print("ANALYSE DÉTAILLÉE DES ERREURS ET POSITIONS")
print(f"{'='*120}\n")

for idx in range(num_examples):
    example = ex[idx]
    context_tokens = example["context_tokens"]
    question_tokens = example["question_tokens"]
    true_start = example["start"]
    true_end = example["end"]
    true_answer = example["raw_answer"]
    
    # Prédiction du modèle
    predicted_answer = predict_answer(model, context_tokens, question_tokens, vocab)
    
    # Calculer les métriques
    em = compute_exact_match(predicted_answer, true_answer)
    f1 = compute_f1_tokens(predicted_answer, true_answer)
    exact_matches += em
    f1_scores.append(f1)
    
    # Obtenir les positions prédites
    ctx = torch.LongTensor([encode(context_tokens, vocab, MAX_CONTEXT_LEN)]).to(device)
    qst = torch.LongTensor([encode(question_tokens, vocab, MAX_QUESTION_LEN)]).to(device)
    
    model.eval()
    with torch.no_grad():
        start_logits, end_logits = model(ctx, qst)
        pred_start = torch.argmax(start_logits[0]).item()
        pred_end = torch.argmax(end_logits[0]).item()
    
    print(f"Exemple {idx + 1}:")
    print(f"Question: {' '.join(question_tokens)}")
    print(f"Vraie réponse: '{true_answer}' (tokens {true_start} à {true_end})")
    print(f"Réponse prédite: '{predicted_answer}' (tokens {pred_start} à {pred_end})")
    print(f"Exact Match: {'✓' if em else '✗'} | F1 Score: {f1:.3f}")
    print(f"Contexte autour de la vraie réponse:")
    if true_start > 0:
        print(f"  Avant: ...{' '.join(context_tokens[max(0, true_start-2):true_start])}")
    print(f"  Réponse: {' '.join(context_tokens[true_start:true_end+1])}")
    if true_end < len(context_tokens) - 1:
        print(f"  Après: {' '.join(context_tokens[true_end+1:min(len(context_tokens), true_end+3)])}...")
    print(f"{'-'*120}\n")

print(f"{'='*120}")
print(f"MÉTRIQUES GLOBALES:")
print(f"Exact Match Accuracy: {exact_matches}/{num_examples} ({100*exact_matches/num_examples:.1f}%)")
print(f"F1 Score moyen: {sum(f1_scores)/len(f1_scores):.3f}")
print(f"{'='*120}\n")

# Recommandations
print("RECOMMANDATIONS POUR AMÉLIORER LE MODÈLE:")
print("-" * 120)
print("1. AUGMENTER LES EPOCHS: Le modèle a seulement 10 epochs. Essayez 30-50 epochs.")
print("2. AJUSTER L'ARCHITECTURE: Ajouter de l'attention et une meilleure fusion context-question")
print("3. AUGMENTER LES DONNÉES: Plus de données d'entraînement = meilleures performances")
print("4. HYPERPARAMETERS: Augmenter embed_dim (200), lstm_units (256), learning rate, ajouter dropout")
print("5. PRE-TRAINED EMBEDDINGS: Utiliser word2vec ou GloVe au lieu d'embeddings aléatoires")
print("6. VALIDATION SET: Évaluer sur un validation set pour tuner les hyperparameters")
print(f"{'='*120}")



ANALYSE DÉTAILLÉE DES ERREURS ET POSITIONS

Exemple 1:
Question: when did beyonce start becoming popular?
Vraie réponse: 'in the late 1990s' (tokens 39 à 42)
Réponse prédite: 'dangerously in love (2003),' (tokens 79 à 82)
Exact Match: ✗ | F1 Score: 0.250
Contexte autour de la vraie réponse:
  Avant: ...to fame
  Réponse: in the late 1990s
  Après: as lead...
------------------------------------------------------------------------------------------------------------------------

Exemple 2:
Question: what areas did beyonce compete in when she was growing up?
Vraie réponse: 'singing and dancing' (tokens 28 à 30)
Réponse prédite: 'late 1990s as lead singer of r&b girl-group destiny's child. managed by her father, mathew knowles, the group became one of the world's best-selling girl groups of all time. their hiatus saw the release of beyoncé's debut album, dangerously in love (2003),' (tokens 41 à 82)
Exact Match: ✗ | F1 Score: 0.000
Contexte autour de la vraie réponse:
  Avant: ...in vari

## Modèle amélioré avec plus d'entraînement

In [38]:
# Modèle amélioré avec Attention
class ImprovedQAModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_units, max_context_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # LSTMs bidirectionnels
        self.lstm_context = nn.LSTM(
            embed_dim, lstm_units, bidirectional=True, batch_first=True, dropout=0.3
        )
        self.lstm_question = nn.LSTM(
            embed_dim, lstm_units, bidirectional=True, batch_first=True, dropout=0.3
        )
        
        # Attention: calcule l'importance de chaque token du contexte par rapport à la question
        self.attention = nn.MultiheadAttention(
            embed_dim=lstm_units * 2, num_heads=4, batch_first=True, dropout=0.1
        )
        
        # Couches denses avec dropout
        self.dense1 = nn.Linear(lstm_units * 4, lstm_units * 2)
        self.dropout = nn.Dropout(0.3)
        self.dense_start = nn.Linear(lstm_units * 2, 1)
        self.dense_end = nn.Linear(lstm_units * 2, 1)
        self.relu = nn.ReLU()
    
    def forward(self, context, question):
        context_emb = self.embedding(context)
        question_emb = self.embedding(question)
        
        # Encode avec LSTMs
        context_enc, _ = self.lstm_context(context_emb)
        question_enc, _ = self.lstm_question(question_emb)
        
        # Attention: utiliser la question comme query et context comme key/value
        attn_output, _ = self.attention(
            context_enc, question_enc, question_enc
        )
        
        # Fusionner context encoding et attention output
        merged = torch.cat([context_enc, attn_output], dim=-1)
        
        # Couches denses
        hidden = self.relu(self.dense1(merged))
        hidden = self.dropout(hidden)
        
        # Prédictions
        start_logits = self.dense_start(hidden).squeeze(-1)
        end_logits = self.dense_end(hidden).squeeze(-1)
        
        return start_logits, end_logits

# Créer le modèle amélioré
improved_model = ImprovedQAModel(VOCAB_SIZE, EMBED_DIM, LSTM_UNITS, MAX_CONTEXT_LEN)
improved_model.to(device)

# Entraîner pendant plus d'epochs
IMPROVED_EPOCHS = 30

optimizer_improved = optim.Adam(improved_model.parameters(), lr=0.001)
criterion_improved = nn.CrossEntropyLoss()

print("Entraînement du modèle amélioré avec attention...")
print("="*80)

improved_model.train()
for epoch in range(IMPROVED_EPOCHS):
    total_loss = 0
    for context, question, start_pos, end_pos in train_loader:
        context = context.to(device)
        question = question.to(device)
        start_pos = start_pos.to(device)
        end_pos = end_pos.to(device)
        
        start_logits, end_logits = improved_model(context, question)
        
        loss_start = criterion_improved(start_logits, start_pos)
        loss_end = criterion_improved(end_logits, end_pos)
        loss = loss_start + loss_end
        
        optimizer_improved.zero_grad()
        loss.backward()
        optimizer_improved.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{IMPROVED_EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("="*80)
torch.save(improved_model.state_dict(), "bidaf_model_improved.pt")
print("Modèle amélioré sauvegardé!")


/Users/albanecoiffe/Downloads/NLP/.venv/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Entraînement du modèle amélioré avec attention...
Epoch 5/30, Loss: 3.2136
Epoch 10/30, Loss: 1.0340
Epoch 15/30, Loss: 0.5535
Epoch 20/30, Loss: 0.3950
Epoch 25/30, Loss: 0.3221
Epoch 30/30, Loss: 0.2799
Modèle amélioré sauvegardé!


## Évaluation du modèle amélioré

In [40]:
def predict_answer_v2(model, context_tokens, question_tokens, vocab):
    """Prédiction avec le modèle amélioré"""
    model.eval()
    with torch.no_grad():
        ctx = torch.LongTensor([encode(context_tokens, vocab, MAX_CONTEXT_LEN)]).to(device)
        qst = torch.LongTensor([encode(question_tokens, vocab, MAX_QUESTION_LEN)]).to(device)
        
        start_logits, end_logits = model(ctx, qst)
        
        start = torch.argmax(start_logits[0]).item()
        end = torch.argmax(end_logits[0]).item()
        
        if end < start:
            end = start
    
    return " ".join(context_tokens[start:end+1]), start, end

# Évaluation du modèle amélioré
num_examples = min(15, len(ex))
exact_matches_improved = 0
f1_scores_improved = []

print(f"\n{'='*120}")
print("ÉVALUATION DU MODÈLE AMÉLIORÉ AVEC ATTENTION")
print(f"{'='*120}\n")

for idx in range(num_examples):
    example = ex[idx]
    context_tokens = example["context_tokens"]
    question_tokens = example["question_tokens"]
    true_answer = example["raw_answer"]
    
    # Prédiction avec le modèle amélioré
    predicted_answer, pred_start, pred_end = predict_answer_v2(
        improved_model, context_tokens, question_tokens, vocab
    )
    
    # Métriques
    em = compute_exact_match(predicted_answer, true_answer)
    f1 = compute_f1_tokens(predicted_answer, true_answer)
    exact_matches_improved += em
    f1_scores_improved.append(f1)
    
    # Affichage
    status = "✓" if em else "✗"
    print(f"{status} Exemple {idx + 1} | F1: {f1:.3f}")
    print(f"   Q: {' '.join(question_tokens[:15])}{'...' if len(question_tokens) > 15 else ''}")
    print(f"   True: '{true_answer}'")
    print(f"   Pred: '{predicted_answer}'")
    print()

print(f"{'='*120}")
print(f"RÉSULTATS DU MODÈLE AMÉLIORÉ:")
print(f"Exact Match: {exact_matches_improved}/{num_examples} ({100*exact_matches_improved/num_examples:.1f}%)")
print(f"F1 Score moyen: {sum(f1_scores_improved)/len(f1_scores_improved):.3f}")
print(f"\nCOMPARAISON:")
print(f"  Modèle initial: 0.0% Exact Match")
print(f"  Modèle amélioré: {100*exact_matches_improved/num_examples:.1f}% Exact Match")
print(f"{'='*120}")



ÉVALUATION DU MODÈLE AMÉLIORÉ AVEC ATTENTION

✓ Exemple 1 | F1: 1.000
   Q: when did beyonce start becoming popular?
   True: 'in the late 1990s'
   Pred: 'in the late 1990s'

✓ Exemple 2 | F1: 1.000
   Q: what areas did beyonce compete in when she was growing up?
   True: 'singing and dancing'
   Pred: 'singing and dancing'

✗ Exemple 3 | F1: 0.000
   Q: when did beyonce leave destiny's child and become a solo singer?
   True: '2003'
   Pred: '(2003),'

✗ Exemple 4 | F1: 0.500
   Q: in what city and state did beyonce grow up?
   True: 'Houston, Texas'
   Pred: 'houston, texas,'

✓ Exemple 5 | F1: 1.000
   Q: in which decade did beyonce become famous?
   True: 'late 1990s'
   Pred: 'late 1990s'

✗ Exemple 6 | F1: 0.500
   Q: in what r&b group was she the lead singer?
   True: 'Destiny's Child'
   Pred: 'destiny's child.'

✓ Exemple 7 | F1: 1.000
   Q: what album made her a worldwide known artist?
   True: 'Dangerously in Love'
   Pred: 'dangerously in love'

✗ Exemple 8 | F1: 0.500
  

## Améliorations complémentaires : Nettoyage et post-traitement

In [41]:
import re
import string

def clean_prediction(prediction):
    """
    Nettoie la prédiction en supprimant la ponctuation inutile
    """
    # Enlever la ponctuation en début et fin
    prediction = re.sub(r'^[\s\(\)\[\],;:\.\"\']+', '', prediction)
    prediction = re.sub(r'[\s\(\)\[\],;:\.\"\']+$', '', prediction)
    # Enlever les espaces multiples
    prediction = re.sub(r'\s+', ' ', prediction).strip()
    return prediction

# Tokenizer amélioré qui gère la ponctuation
def tokenize_advanced(text):
    """
    Tokenization améliorée qui sépare la ponctuation
    """
    text = text.lower()
    # Ajouter des espaces autour de la ponctuation
    text = re.sub(r'([.,!?;:\'\"\(\)\[\]])', r' \1 ', text)
    # Nettoyer les espaces multiples
    tokens = text.split()
    return tokens

# Évaluation avec nettoyage
print(f"\n{'='*120}")
print("ÉVALUATION AVEC POST-TRAITEMENT DES PRÉDICTIONS")
print(f"{'='*120}\n")

num_examples = min(15, len(ex))
exact_matches_cleaned = 0
f1_scores_cleaned = []
improvements = 0

for idx in range(num_examples):
    example = ex[idx]
    context_tokens = example["context_tokens"]
    question_tokens = example["question_tokens"]
    true_answer = example["raw_answer"]
    
    # Prédiction brute
    predicted_answer_raw, _, _ = predict_answer_v2(improved_model, context_tokens, question_tokens, vocab)
    
    # Prédiction nettoyée
    predicted_answer_clean = clean_prediction(predicted_answer_raw)
    
    # Métriques
    em_clean = compute_exact_match(predicted_answer_clean, true_answer)
    f1_clean = compute_f1_tokens(predicted_answer_clean, true_answer)
    exact_matches_cleaned += em_clean
    f1_scores_cleaned.append(f1_clean)
    
    # Vérifier si le nettoyage a amélioré
    em_raw = compute_exact_match(predicted_answer_raw, true_answer)
    if em_clean and not em_raw:
        improvements += 1
    
    status = "✓" if em_clean else "✗"
    status_improvement = " (AMÉLIORÉ)" if em_clean and not em_raw else ""
    
    print(f"{status} Exemple {idx + 1} | F1: {f1_clean:.3f}{status_improvement}")
    print(f"   Q: {' '.join(question_tokens[:15])}{'...' if len(question_tokens) > 15 else ''}")
    print(f"   True: '{true_answer}'")
    if predicted_answer_raw != predicted_answer_clean:
        print(f"   Pred (brut): '{predicted_answer_raw}' → (nettoyé): '{predicted_answer_clean}'")
    else:
        print(f"   Pred: '{predicted_answer_clean}'")
    print()

print(f"{'='*120}")
print(f"RÉSULTATS AVEC POST-TRAITEMENT:")
print(f"Exact Match: {exact_matches_cleaned}/{num_examples} ({100*exact_matches_cleaned/num_examples:.1f}%)")
print(f"F1 Score moyen: {sum(f1_scores_cleaned)/len(f1_scores_cleaned):.3f}")
print(f"Prédictions améliorées par nettoyage: {improvements}")
print(f"\nCOMPARAISON DES TROIS MODÈLES:")
print(f"  1. Modèle initial (10 epochs):        0.0% Exact Match")
print(f"  2. Modèle amélioré (30 epochs):      66.7% Exact Match")
print(f"  3. Modèle + nettoyage:               {100*exact_matches_cleaned/num_examples:.1f}% Exact Match")
print(f"{'='*120}\n")



ÉVALUATION AVEC POST-TRAITEMENT DES PRÉDICTIONS

✓ Exemple 1 | F1: 1.000
   Q: when did beyonce start becoming popular?
   True: 'in the late 1990s'
   Pred: 'in the late 1990s'

✓ Exemple 2 | F1: 1.000
   Q: what areas did beyonce compete in when she was growing up?
   True: 'singing and dancing'
   Pred: 'singing and dancing'

✓ Exemple 3 | F1: 1.000 (AMÉLIORÉ)
   Q: when did beyonce leave destiny's child and become a solo singer?
   True: '2003'
   Pred (brut): '(2003),' → (nettoyé): '2003'

✓ Exemple 4 | F1: 1.000 (AMÉLIORÉ)
   Q: in what city and state did beyonce grow up?
   True: 'Houston, Texas'
   Pred (brut): 'houston, texas,' → (nettoyé): 'houston, texas'

✓ Exemple 5 | F1: 1.000
   Q: in which decade did beyonce become famous?
   True: 'late 1990s'
   Pred: 'late 1990s'

✓ Exemple 6 | F1: 1.000 (AMÉLIORÉ)
   Q: in what r&b group was she the lead singer?
   True: 'Destiny's Child'
   Pred (brut): 'destiny's child.' → (nettoyé): 'destiny's child'

✓ Exemple 7 | F1: 1.000
   